# $\Pi_0^1$ projector — load a Netgen `.vol` mesh

This is a **separate application-only demo** showing the workflow

```python
mesh = Mesh("sample_fichera.vol")
P = Pi1Projector(mesh)
Pi1_u = P(u)
```

The implementation is in `pi1projector.py`. This notebook does **not** run convergence studies, projection-error studies, commuting checks, or verification tests.

**Assumption:** `sample_fichera.vol` is a **straight-sided tetrahedral volume mesh** compatible with the present projector.


In [ ]:
# Install NGSolve in Colab if needed.
import importlib.util, subprocess, sys
if importlib.util.find_spec("ngsolve") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "ngsolve==6.2.2606", "webgui_jupyter_widgets"
    ])


## Load the projector module and sample `.vol` mesh

The bundle includes `sample_fichera.vol`, a coarse straight-sided tetrahedral Fichera-corner mesh in native Netgen `.vol` format.

If these files are not already in the Colab working directory, upload:

- `pi1projector.py`
- `sample_fichera.vol`

You can later replace `sample_fichera.vol` by any supported user `.vol` mesh.


In [ ]:
import os
from google.colab import files

missing = [name for name in ("pi1projector.py", "sample_fichera.vol") if not os.path.exists(name)]
if missing:
    print("Upload the missing bundle file(s):", missing)
    files.upload()

from pi1projector import Pi1Projector
from ngsolve import *
from ngsolve.webgui import Draw


## Load the mesh

If your uploaded file has a different name, replace `"sample_fichera.vol"` below.


In [ ]:
mesh_filename = "sample_fichera.vol"   # change this if needed
mesh = Mesh(mesh_filename)

print("loaded mesh:", mesh_filename)
print("volume tetrahedra =", len(list(mesh.Elements(VOL))))
Draw(mesh)


## Define a vector field and project it

`u` must be a **three-component** NGSolve `CoefficientFunction`.


In [ ]:
u = CF((
    x*x + y,
    y*y + z,
    z*z + x,
))

P = Pi1Projector(mesh)
Pi1_u = P(u)

Pi1_u


## Manipulate the projected field

`Pi1_u` is an ordinary NGSolve `HCurl` `GridFunction` on the original loaded mesh.


In [ ]:
curl_Pi1_u = curl(Pi1_u)

# Example visualization of a Cartesian volume component on the boundary.
Draw(BoundaryFromVolumeCF(Pi1_u)[0], mesh, "Pi1_u_x")


## Reuse the same projector

The expensive projector construction is reused for every additional field on the same mesh.


In [ ]:
v = CF((y*z, z*x, x*y))
Pi1_v = P(v, name="Pi1_v")
